# Churn — nettoyage de la table finale

Ce notebook prend `data/final/all_tables.csv` et le nettoie pour le modèle.

**Entrée** : 34 000 lignes × 10 colonnes
**Sortie** : `data/final/all_tables_clean.csv`

## Les 7 étapes

| # | Étape | Décision |
|---|---|---|
| 1 | Valeurs manquantes | remplir à 0 |
| 2 | Lignes dupliquées | conserver (6 contrats distincts) |
| 3 | Valeurs extrêmes | ne rien faire (mesuré : aucun effet) |
| 4 | `OFFRE` : 65 modalités | garder brute (mesuré : le regroupement est moins bon) |
| 5 | Encodage des catégories | codes numériques |
| 6 | `STATUT` | garder, mais signaler le biais |
| 7 | Export | `all_tables_clean.csv` |

## 0. Préparation

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SOURCE = ROOT / 'data' / 'final' / 'all_tables.csv'
SORTIE = ROOT / 'data' / 'final' / 'all_tables_clean.csv'

NUMERIQUES = ['ANC_M', 'VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT',
              'APPEL_TOTAL', 'SMS_TOTAL']
CATEGORIELLES = ['HANDSET', 'OFFRE', 'STATUT']

# na_values=[''] : les cellules vides deviennent NaN (sinon invisibles).
# keep_default_na=False : HANDSET='NA' reste une CATEGORIE, pas un manquant.
df = pd.read_csv(SOURCE, keep_default_na=False, na_values=[''], low_memory=False)

for c in NUMERIQUES + ['churn']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

print('données chargées :', df.shape)
df.head(3)

données chargées : (34000, 10)


,ANC_M,HANDSET,OFFRE,STATUT,churn,VOLUME_SESSION,MNT_RECH,MNT_FORFAIT,APPEL_TOTAL,SMS_TOTAL
0,177,4G,PRE - TT 1500%,A,0,5277.095455,21.727273,14.281818,7359.43,1768.0
1,177,2G,PRE - Double,A,0,NaN,5.222222,0.800000,598.95,592.0
2,178,4G,PRE - TT 1500%,A,0,5242.772727,27.727273,1.737500,10223.95,2290.0


## 1. Valeurs manquantes

### 1.1 Diagnostic

Combien de vides, et sont-ils liés au churn ?

In [2]:
manquants = df.isna().sum()
manquants = manquants[manquants > 0]

print('colonne           vides      %    churn(si vide)  churn(si présent)')
print('-' * 72)
for c in manquants.index:
    absent = df[c].isna()
    taux_absent = df.loc[absent, 'churn'].mean() * 100
    taux_present = df.loc[~absent, 'churn'].mean() * 100
    print(f'{c:16s} {absent.sum():>6,}  {absent.mean()*100:>5.1f}%'
          f'      {taux_absent:>5.1f}%           {taux_present:>5.1f}%')

colonne           vides      %    churn(si vide)  churn(si présent)
------------------------------------------------------------------------
VOLUME_SESSION    9,729   28.6%       30.1%            27.3%
MNT_RECH            217    0.6%       44.7%            28.0%
MNT_FORFAIT      10,628   31.3%       30.3%            27.1%


**Lecture** : les clients qui ont un vide churnent **plus** que les autres. Le vide n'est pas neutre — c'est un signal.

Mais attention : un vide n'a pas le même sens partout.

| Situation | Signification |
|---|---|
| **vide** | le client n'est **pas du tout** dans cette table → aucune activité de ce type |
| **0** | le client est dans la table, mais la valeur mesurée est nulle |

Ce ne sont pas les mêmes clients. Vérifions :

In [3]:
print('VOLUME_SESSION — trois populations différentes :')
print()

absent = df['VOLUME_SESSION'].isna()
zero = (df['VOLUME_SESSION'] == 0)
positif = ~absent & ~zero

for nom, masque in [('vide (absent de la table)', absent),
                    ('0 mesuré', zero),
                    ('> 0', positif)]:
    n = int(masque.sum())
    taux = df.loc[masque, 'churn'].mean() * 100
    print(f'  {nom:26s} {n:>6,} contrats   churn {taux:>5.1f} %')

print()
print('Les trois groupes ont des taux de churn différents :')
print('le vide porte une information que 0 ne porte pas.')

VOLUME_SESSION — trois populations différentes :

  vide (absent de la table)   9,729 contrats   churn  30.1 %
  0 mesuré                      821 contrats   churn  22.5 %
  > 0                        23,450 contrats   churn  27.5 %

Les trois groupes ont des taux de churn différents :
le vide porte une information que 0 ne porte pas.


### 1.2 Les 5 stratégies possibles

| # | Stratégie | Code | Principe |
|---|---|---|---|
| 1 | Ne rien faire | — | le gradient boosting gère les NaN nativement |
| 2 | Remplir à 0 | `.fillna(0)` | tous les vides deviennent 0 |
| 3 | 0 + marqueur par mesure | `+ 3 colonnes` | garde l'info du vide par colonne |
| 4 | 0 + un marqueur global | `+ a_data` | garde l'info du vide, une seule colonne |
| 5 | Médiane | `.fillna(median)` | suppose que l'absent ressemble au client moyen |

### 1.3 Comparaison mesurée

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

A_REMPLIR = ['VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT']

def encoder_categories(table):
    """Transforme les colonnes texte en codes numériques."""
    resultat = table[NUMERIQUES].copy()
    for c in CATEGORIELLES:
        resultat[c] = df[c].astype('category').cat.codes
    return resultat

def evaluer(X, label):
    """AUC en validation croisée à 5 plis."""
    scores = cross_val_score(
        HistGradientBoostingClassifier(max_iter=250, random_state=0),
        X, df['churn'], cv=StratifiedKFold(5, shuffle=True, random_state=0),
        scoring='roc_auc', n_jobs=-1)
    print(f'{label:46s} AUC = {scores.mean():.5f}')
    return scores.mean()

resultats = {}

# 1. ne rien faire
resultats['rien'] = evaluer(encoder_categories(df), '1. ne rien faire')

# 2. remplir à 0
d2 = df.copy()
d2[A_REMPLIR] = d2[A_REMPLIR].fillna(0)
resultats['zero'] = evaluer(encoder_categories(d2), '2. remplir à 0')

# 3. 0 + un marqueur par mesure
d3 = d2.copy()
X3 = encoder_categories(d3)
for c in A_REMPLIR:
    X3[c + '_absent'] = df[c].isna().astype(int)
resultats['marqueurs'] = evaluer(X3, '3. 0 + marqueur par mesure')

# 4. 0 + un seul marqueur global
d4 = d2.copy()
X4 = encoder_categories(d4)
X4['a_data'] = (~df['VOLUME_SESSION'].isna()).astype(int)
resultats['global'] = evaluer(X4, '4. 0 + 1 marqueur global')

# 5. médiane
d5 = df.copy()
for c in A_REMPLIR:
    d5[c] = d5[c].fillna(df[c].median())
resultats['mediane'] = evaluer(encoder_categories(d5), '5. remplir par la médiane')

print()
print('--- classement ---')
for nom, score in sorted(resultats.items(), key=lambda x: -x[1]):
    print(f'  {nom:12s} {score:.5f}')
print()
print('Les 5 stratégies sont séparées par moins de 0,001 d AUC :')
print('elles se valent toutes. On choisit la plus simple : remplir à 0.')

1. ne rien faire                               AUC = 0.92090


2. remplir à 0                                 AUC = 0.92119


3. 0 + marqueur par mesure                     AUC = 0.92118


4. 0 + 1 marqueur global                       AUC = 0.92144


5. remplir par la médiane                      AUC = 0.92111

--- classement ---
  global       0.92144
  zero         0.92119
  marqueurs    0.92118
  mediane      0.92111
  rien         0.92090

Les 5 stratégies sont séparées par moins de 0,001 d AUC :
elles se valent toutes. On choisit la plus simple : remplir à 0.


## 2. Lignes dupliquées

### 2.1 Diagnostic

In [7]:
dupliquees = df_clean.duplicated(keep=False)
print('lignes impliquées dans un doublon :', int(dupliquees.sum()))
print()
if dupliquees.any():
    print(df_clean[dupliquees].to_string(index=False))
print()
print('--- ces lignes sont-elles de VRAIS doublons ? ---')
# all_tables.csv est construit par left join depuis agg_cible,
# qui contient exactement 1 ligne par CODE_CONTRAT.
cible = pd.read_csv(ROOT / 'data' / 'processed' / 'agg_cible.csv',
                    keep_default_na=False)
print('contrats dans agg_cible :', len(cible),
      '| CODE_CONTRAT uniques :', cible['CODE_CONTRAT'].nunique())
print()
print('Chaque ligne du fichier correspond donc à un contrat DIFFÉRENT.')
print('Des valeurs identiques ne signifient pas le même client :')
print('ce sont des contrats distincts qui ont le même profil (tout à 0).')

lignes impliquées dans un doublon : 6

 ANC_M HANDSET                      OFFRE STATUT  churn  VOLUME_SESSION  MNT_RECH  MNT_FORFAIT  APPEL_TOTAL  SMS_TOTAL
     4      NA           PRE - Trankil TT      A      0             0.0       0.0          0.0          0.0        0.0
     4      4G PRE - ESS Mobile 35mil/min      A      0             0.0       0.0          0.0          0.0        0.0
     4      NA           PRE - Trankil TT      A      0             0.0       0.0          0.0          0.0        0.0
     4      4G PRE - ESS Mobile 35mil/min      A      0             0.0       0.0          0.0          0.0        0.0
     3      NA       PRE - CSSM 35mil/min      A      0             0.0       0.0          0.0          0.0        0.0
     3      NA       PRE - CSSM 35mil/min      A      0             0.0       0.0          0.0          0.0        0.0

--- ces lignes sont-elles de VRAIS doublons ? ---
contrats dans agg_cible : 34000 | CODE_CONTRAT uniques : 34000

Chaque ligne 

## 3. Valeurs extrêmes

### 3.1 Diagnostic

Certaines valeurs sont très au-dessus du reste. Sont-elles des erreurs, ou de vrais gros clients ?

In [9]:
print('colonne           P99       P99.9        max    max/P99   nb > 3xP99')
print('-' * 72)
for c in ['VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT', 'APPEL_TOTAL', 'SMS_TOTAL']:
    s = df_clean[c]
    p99 = s.quantile(0.99)
    p999 = s.quantile(0.999)
    mx = s.max()
    ratio = mx / p99 if p99 > 0 else np.nan
    nb = int((s > 3 * p99).sum()) if p99 > 0 else 0
    print(f'{c:16s} {p99:>9.1f} {p999:>9.1f} {mx:>10.1f} {ratio:>8.1f} {nb:>10,}')

colonne           P99       P99.9        max    max/P99   nb > 3xP99
------------------------------------------------------------------------
VOLUME_SESSION     45487.4   82687.2   194047.2      4.3          6
MNT_RECH              75.1     142.2     2451.5     32.6          5
MNT_FORFAIT           57.2     103.5     1223.0     21.4          4
APPEL_TOTAL        18568.6   34390.9    67259.2      3.6          2
SMS_TOTAL           4887.0   10347.3    49391.0     10.1         16


**Observation** : `MNT_RECH` monte à 32 fois le P99 (2 451 DT contre 75 DT). `SMS_TOTAL` à 10 fois.

Ce ne sont probablement **pas des erreurs** : ce sont de vrais clients à très forte consommation (professionnels, cybercafés, hotspots). Les supprimer reviendrait à supprimer les meilleurs clients.

### 3.2 Comparaison mesurée

In [10]:
print('stratégie                  AUC')
print('-' * 34)

for q, label in [(1.0, 'aucun plafonnement'),
                 (0.99, 'plafonner au P99'),
                 (0.999, 'plafonner au P99.9')]:
    d_test = df_clean.copy()
    if q < 1.0:
        for c in ['VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT',
                  'APPEL_TOTAL', 'SMS_TOTAL']:
            d_test[c] = d_test[c].clip(upper=d_test[c].quantile(q))
    X = pd.DataFrame(d_test[NUMERIQUES])
    for c in CATEGORIELLES:
        X[c] = d_test[c].astype('category').cat.codes
    score = cross_val_score(
        HistGradientBoostingClassifier(max_iter=250, random_state=0),
        X, d_test['churn'],
        cv=StratifiedKFold(5, shuffle=True, random_state=0),
        scoring='roc_auc', n_jobs=-1).mean()
    print(f'{label:26s} {score:.5f}')

stratégie                  AUC
----------------------------------


aucun plafonnement         0.92119


plafonner au P99           0.92160


plafonner au P99.9         0.92119


## 4. La colonne `OFFRE` — 65 modalités

### 4.1 Diagnostic

Avec 65 modalités, un modèle risque de mémoriser des offres vues une seule fois au lieu d'apprendre. Mais combien de contrats sont concernés ?

In [12]:
compte = df_clean['OFFRE'].value_counts()
print(f'nombre de modalités : {len(compte)}')
print()
print('Les 10 plus fréquentes :')
print(compte.head(10).to_string())
print()
print(f'offres avec moins de 50 contrats : {int((compte < 50).sum())}')
print(f'contrats concernés               : {int(compte[compte < 50].sum()):,}  '
      f'({compte[compte < 50].sum() / len(df_clean) * 100:.1f} %)')
print()
cumul = (compte.cumsum() / len(df_clean)).to_numpy()
for seuil in [0.80, 0.90, 0.95]:
    n = int((cumul < seuil).sum()) + 1
    print(f'{int(seuil*100)} % des contrats couverts par {n} offres sur {len(compte)}')

nombre de modalités : 65

Les 10 plus fréquentes :
OFFRE
PRE - Trankil TT        13892
PRE - 1=11               3836
PRE - TT 1500%           2899
PRE - TT 1000%           1760
Hayya                    1644
PRE - 900 bonus          1236
PRE - Double             1010
100% Bonus_TT             978
PRE - TM 35mil/min        741
PRE - Family Connect      665

offres avec moins de 50 contrats : 29
contrats concernés               : 466  (1.4 %)

80 % des contrats couverts par 8 offres sur 65
90 % des contrats couverts par 15 offres sur 65
95 % des contrats couverts par 23 offres sur 65


In [13]:
print('Les libellés montrent des doublons de saisie :')
for libelle in df_clean['OFFRE'].unique():
    if '%%' in libelle or '?' in libelle or 'REINSTAL' in libelle.upper():
        n = int((df_clean['OFFRE'] == libelle).sum())
        print(f'  {libelle:32s} {n:>5,} contrats')

Les libellés montrent des doublons de saisie :
  PRE - Double Reinstal              246 contrats
  PRE - CSS Mobile 1500%%             21 contrats
  PRE - TAWWA REINSTALLED             11 contrats
  PRE - Binetna Reinstal               5 contrats
  PRE ? 100% EM                        9 contrats


### 4.2 Les deux options

**Option A — garder les 65 libellés bruts.** Aucune perte d'information, mais beaucoup de modalités rares.

**Option B — regrouper en familles.** Moins de modalités, plus lisible, mais on perd les nuances.

### 4.3 Comparaison mesurée

In [14]:
def famille_offre(libelle):
    """Regroupe un libellé d'offre en famille."""
    t = re.sub(r'\s+', ' ', libelle.upper()).strip()
    if 'TRANKIL' in t:                     return 'Trankil'
    if 'HAYYA' in t:                       return 'Hayya'
    if 'FAMILY' in t:                      return 'Family'
    if 'BONUS' in t:                       return 'Bonus'
    if '1=11' in t:                        return '1=11'
    if re.search(r'\d{3,4}\s*%', t):        return 'Pourcentage'
    if '35MIL' in t or '35MI' in t:        return 'Illimite'
    return 'Autre'

familles = df_clean['OFFRE'].map(famille_offre)
print('regroupement :', df_clean['OFFRE'].nunique(), 'libellés ->', familles.nunique(), 'familles')
print()
print(familles.value_counts().to_string())
print()

for nom, serie in [('A. 65 libellés bruts', df_clean['OFFRE']),
                   ('B. familles regroupées', familles)]:
    X = pd.DataFrame(df_clean[NUMERIQUES])
    X['HANDSET'] = df_clean['HANDSET'].astype('category').cat.codes
    X['STATUT'] = df_clean['STATUT'].astype('category').cat.codes
    X['OFFRE'] = serie.astype('category').cat.codes
    score = cross_val_score(
        HistGradientBoostingClassifier(max_iter=250, random_state=0),
        X, df_clean['churn'],
        cv=StratifiedKFold(5, shuffle=True, random_state=0),
        scoring='roc_auc', n_jobs=-1).mean()
    print(f'{nom:26s} AUC = {score:.5f}')

regroupement : 65 libellés -> 8 familles

OFFRE
Trankil        14072
Pourcentage     6540
1=11            3839
Autre           2534
Bonus           2488
Illimite        2218
Hayya           1644
Family           665



A. 65 libellés bruts       AUC = 0.92119


B. familles regroupées     AUC = 0.92133


## 5. Encodage des variables catégorielles

Les modèles ne savent pas lire du texte : il faut transformer `4G`, `Hayya`, `A` en nombres.

### 5.1 Les deux méthodes

| Méthode | Principe | Avantage | Inconvénient |
|---|---|---|---|
| **Codes numériques** | `4G` → 2, `5G` → 3 | compact, 1 colonne | crée un ordre artificiel (5G > 4G > 3G) |
| **One-hot** | 4 colonnes binaires | pas d'ordre artificiel | beaucoup de colonnes avec 65 offres |

### 5.2 Décision : codes numériques

Pourquoi ça marche malgré l'ordre artificiel ? Parce qu'un **arbre de décision** ne fait pas d'arithmétique sur les codes : il teste seulement `OFFRE <= 12`, ce qui revient à isoler un sous-groupe. Il peut donc reconstruire l'information sans être trompé par l'ordre.

Pour un modèle **linéaire**, ce serait faux : il multiplierait le code par un coefficient, comme si `5G` valait trois fois `3G`.

In [15]:
df_encode = df_clean.copy()

for c in CATEGORIELLES:
    df_encode[c] = df_encode[c].astype('category').cat.codes

print('encodage appliqué :')
for c in CATEGORIELLES:
    n = df_clean[c].nunique()
    print(f'  {c:10s} {n:>3} modalités -> codes 0 à {n-1}')
print()
df_encode[CATEGORIELLES].head(5)

encodage appliqué :
  HANDSET      5 modalités -> codes 0 à 4
  OFFRE       65 modalités -> codes 0 à 64
  STATUT       2 modalités -> codes 0 à 1



,HANDSET,OFFRE,STATUT
0,2,51,0
1,0,21,0
2,2,51,0
3,0,62,0
4,0,51,0


## 6. La question `STATUT` — à décider en connaissance de cause

### 6.1 Pourquoi c'est un problème

`STATUT` est l'état **actuel** du contrat : `A` (actif) ou `S` (suspendu).

| STATUT | Contrats | Churn |
|---|---|---|
| A | 25 466 | **9,8 %** |
| S | 8 534 | **82,9 %** |

Une seule colonne binaire fait passer l'AUC de 0,81 à 0,92. Mais **est-ce de la prédiction ?**

Un client déjà suspendu est déjà perdu dans les faits. Le modèle apprend « S reste S » — ce n'est pas faux, mais ça n'aide pas à identifier les clients **actifs** qui vont partir, qui est l'usage métier réel.

In [16]:
X_sans = pd.DataFrame(df_encode[NUMERIQUES])
X_sans['HANDSET'] = df_encode['HANDSET']
X_sans['OFFRE'] = df_encode['OFFRE']

X_avec = X_sans.copy()
X_avec['STATUT'] = df_encode['STATUT']

for nom, X in [('sans STATUT', X_sans), ('avec STATUT', X_avec)]:
    score = cross_val_score(
        HistGradientBoostingClassifier(max_iter=250, random_state=0),
        X, df_clean['churn'],
        cv=StratifiedKFold(5, shuffle=True, random_state=0),
        scoring='roc_auc', n_jobs=-1).mean()
    print(f'{nom:16s} AUC = {score:.5f}')

sans STATUT      AUC = 0.80843


avec STATUT      AUC = 0.92119


### 6.2 Décision : on garde `STATUT`

| Modèle | Colonnes | AUC | Usage |
|---|---|---|---|
| Opérationnel | avec `STATUT` | 0,921 | prioriser les actions du mois |
| Comportemental | sans `STATUT` | 0,809 | comprendre et prévenir |

## 7. Export

In [18]:
# version complète (toutes les colonnes, encodées)
df_encode.to_csv(SORTIE, index=False)

# les deux versions prêtes pour le modèle
version_comportementale.to_csv(ROOT / 'data' / 'final' / 'modele_comportement.csv', index=False)
version_operationnelle.to_csv(ROOT / 'data' / 'final' / 'modele_operationnel.csv', index=False)

print('fichiers écrits dans data/final/ :')
for nom in ['all_tables_clean.csv', 'modele_comportement.csv', 'modele_operationnel.csv']:
    f = ROOT / 'data' / 'final' / nom
    d = pd.read_csv(f, nrows=1, keep_default_na=False)
    print(f'  {nom:26s} {len(d.columns):>3} colonnes  ({f.stat().st_size / 1024:>7.0f} Ko)')

fichiers écrits dans data/final/ :
  all_tables_clean.csv        10 colonnes  (   1873 Ko)
  modele_comportement.csv      9 colonnes  (   1807 Ko)
  modele_operationnel.csv     10 colonnes  (   1873 Ko)


In [19]:
# contrôle final
verif = pd.read_csv(SORTIE, keep_default_na=False, na_values=[''], low_memory=False)
print('=== contrôle ===')
print('forme              :', verif.shape)
print('valeurs manquantes :', int(verif.isna().sum().sum()))
print('doublons           :', int(verif.duplicated().sum()),
      '(<-- 3 paires de contrats DISTINCTS, conservées volontairement)')
print('taux de churn      :', round(verif['churn'].mean() * 100, 2), '%')
print()
print('colonnes :', list(verif.columns))

=== contrôle ===
forme              : (34000, 10)
valeurs manquantes : 0
doublons           : 3 (<-- 3 paires de contrats DISTINCTS, conservées volontairement)
taux de churn      : 28.12 %

colonnes : ['ANC_M', 'HANDSET', 'OFFRE', 'STATUT', 'churn', 'VOLUME_SESSION', 'MNT_RECH', 'MNT_FORFAIT', 'APPEL_TOTAL', 'SMS_TOTAL']
